# Transformer-XL — Attentive Language Models Beyond a Fixed-Length Context (2019)

_Papers · Transformers & LLMs_

**Cache the previous segment's hidden states and reuse them as extra context, plus a relative position scheme, so a transformer's effective context stretches far beyond one fixed-length window.**

---

This notebook is a practice scaffold for the **Transformer-XL — Attentive Language Models Beyond a Fixed-Length Context (2019)** lesson. We build it up one piece at a time: a worked attention-with-memory example, the XL self-attention layer, a full block, then a tiny model on a cross-segment task with a clean ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Hand-check attention with a cached segment

The core idea is that the query from the *current* segment can attend to keys/values from an *extended* context — the current token concatenated with the previous segment's cached hidden states. With identity projections and `d_k=1`, we work one step by hand: a single current token (hidden state `2`) attends over the extended context `[4, -1, 2]`. The cached value `4` scores highest, so softmax pulls the output toward it — something the vanilla no-memory model (which sees only the current token) can never do. The `.detach()` on the memory is the stop-gradient (SG) from §3.2.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- Worked example: single-head attention with memory (identity projections, d_k=1). ---
# Cached previous segment held hidden states [4, -1]; current segment has one token [2].
mem = torch.tensor([[4.], [-1.]])               # 2 cached tokens, 1 feature each
cur = torch.tensor([[2.]])                       # 1 current token (the only query)
ctx = torch.cat([mem.detach(), cur], dim=0)      # extended context = [4, -1, 2]  (SG = detach)

q = cur                                          # queries from CURRENT segment only (\S 3.2)
k = v = ctx                                      # keys/values from the EXTENDED context
scores = (q @ k.transpose(0, 1)).squeeze(0)      # q.k for each key: [8, -2, 4]  (d_k=1, no sqrt)
w = F.softmax(scores, dim=-1)                    # ~ [0.9820, 0.0000, 0.0180]
out_mem = (w @ v).item()                         # ~ 3.964

# Vanilla (no memory): only key/value is the current token itself.
w_vanilla = F.softmax(q @ cur.transpose(0, 1), dim=-1)
out_vanilla = (w_vanilla @ cur).item()           # softmax of one score -> 2.0

print("extended context :", ctx.squeeze(-1).tolist())          # [4.0, -1.0, 2.0]
print("attn weights     :", [round(x, 4) for x in w.tolist()]) # [0.982, 0.0, 0.018]
print("output WITH mem  :", round(out_mem, 3))                 # 3.964  (pulled toward cached 4)
print("output NO mem    :", round(out_vanilla, 3))             # 2.0    (sees only itself)

### Step 2 — Build the XL self-attention layer

Now we generalize that hand calculation into a module. When a memory is supplied, it is detached and prepended to the current segment to form the extended context that keys and values are computed from — while queries still come from the current segment only. On top of the scaled dot-product scores we add a **relative position bias**: a learned scalar per signed query-key distance `i-j`, a tiny stand-in for the paper's `R_{i-j}` term (§3.3). Distance is well-defined across the segment boundary, which is why relative positions and segment recurrence go together.

In [ ]:
# --- XL self-attention: optional memory as extra keys/values + distance-based relative bias. ---
class XLSelfAttention(nn.Module):
    def __init__(self, d_model, max_rel=64):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.d_model = d_model
        # tiny stand-in for R_{i-j}: a learned scalar bias per query-key distance (\S 3.3).
        self.max_rel = max_rel
        self.rel_bias = nn.Parameter(torch.zeros(2 * max_rel + 1))

    def forward(self, x, mem=None):
        # x: (S, d). mem: (M, d) cached previous-segment states, or None.
        if mem is not None:
            ctx = torch.cat([mem.detach(), x], dim=0)   # SG(mem) o x  -> extended context (\S 3.2)
        else:
            ctx = x
        S = x.shape[0]                                   # current segment length
        M = ctx.shape[0] - x.shape[0]                    # memory length
        q = self.W_q(x)                                  # queries: CURRENT segment only
        k = self.W_k(ctx)                                # keys: EXTENDED context
        val = self.W_v(ctx)                              # values: EXTENDED context
        scores = q @ k.transpose(0, 1) / math.sqrt(self.d_model)   # (S, M+S)
        # relative bias indexed by signed distance i-j (current query index vs extended key index).
        qi = torch.arange(S).unsqueeze(1) + M            # current query absolute index in ctx
        kj = torch.arange(M + S).unsqueeze(0)            # every key index in ctx
        dist = (qi - kj).clamp(-self.max_rel, self.max_rel) + self.max_rel
        scores = scores + self.rel_bias[dist]
        attn = F.softmax(scores, dim=-1)
        return self.W_o(attn @ val)                      # (S, d)

### Step 3 — Wrap a block and a segment-processing model

The `XLBlock` is the usual transformer sublayer pattern — attention then feed-forward, each with a residual and LayerNorm — but the attention now takes an optional memory. `TinyXL` processes a list of segments **in order**: each segment is encoded with the previous segment's output as its memory, and then *this* segment's output is cached for the next one. The `use_mem` flag drops the cache, recovering the vanilla segmented baseline used in the ablation.

In [ ]:
# --- One XL block: attention(with memory) + residual/LayerNorm + feed-forward. ---
class XLBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.attn = XLSelfAttention(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.n1 = nn.LayerNorm(d_model)
        self.n2 = nn.LayerNorm(d_model)

    def forward(self, x, mem=None):
        x = self.n1(x + self.attn(x, mem))
        x = self.n2(x + self.ff(x))
        return x


# --- A 1-layer XL model that processes a sequence segment by segment, caching state. ---
class TinyXL(nn.Module):
    def __init__(self, vocab, d_model=32, d_ff=64):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.block = XLBlock(d_model, d_ff)
        self.out = nn.Linear(d_model, vocab)

    def forward(self, segments, use_mem=True):
        # segments: list of (S,) LongTensors, in order. Returns logits per segment.
        mem = None
        logits = []
        for seg in segments:
            h = self.embed(seg)
            h = self.block(h, mem if use_mem else None)   # use_mem=False -> vanilla segmented baseline
            logits.append(self.out(h))
            mem = h                                        # cache THIS segment's output for the next one
        return logits

### Step 4 — The cross-segment echo task, with and without memory

The task plants a CUE token at the start of segment 1; the symbol right after it is the *answer*, which segment 2 must reproduce at its last position. Because the answer lives in the previous segment, only a model that can attend back into the cached segment can solve it. We train twice — memory on, memory off — identical except for that one flag. The XL model climbs to high accuracy; the vanilla baseline plateaus near chance (~1/10) because segment 2 simply cannot see segment 1 (the context-fragmentation problem, §3.1).

In [ ]:
# --- Cross-segment ECHO task: a CUE token in seg1, its 'answer' must be output in seg2. ---
# seg1 = [CUE, a, b, c]; the symbol right after CUE is the ANSWER. seg2 must output ANSWER at its last pos.
# The answer lives in the PREVIOUS segment -> only a model with memory can fetch it.
VOCAB, L, B = 12, 4, 256
CUE = VOCAB - 1

def batch():
    seg1 = torch.randint(1, CUE, (B, L))
    seg1[:, 0] = CUE                                       # plant the cue at the start of seg1
    answer = seg1[:, 1]                                    # the token right after the cue
    seg2 = torch.randint(1, CUE, (B, L))                  # filler current segment
    target2 = seg2.clone()
    target2[:, -1] = answer                                # seg2's last position must reproduce the answer
    return seg1, seg2, target2

def train(use_mem, steps=600, lr=3e-3):
    torch.manual_seed(0)
    net = TinyXL(VOCAB)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lf = nn.CrossEntropyLoss()
    for s in range(steps):
        s1, s2, t2 = batch()
        acc = 0.0
        # process per example (tiny task; clarity over speed)
        opt.zero_grad()
        loss = 0.0
        for b in range(B):
            lg1, lg2 = net([s1[b], s2[b]], use_mem=use_mem)
            loss = loss + lf(lg2, t2[b])
            acc += (lg2[-1].argmax() == t2[b, -1]).float().item()   # accuracy on the cross-segment cue
        (loss / B).backward()
        opt.step()
        if s % 150 == 0 or s == steps - 1:
            print(f"  step {s:4d}  cue-acc {acc / B:.3f}")
    return acc / B

print("\nWITH segment memory (Transformer-XL style, use_mem=True):")
acc_xl = train(use_mem=True)
print("WITHOUT memory (ABLATION = vanilla segmented baseline, use_mem=False):")
acc_van = train(use_mem=False)
print(f"\nfinal cross-segment cue accuracy  XL: {acc_xl:.3f}   vanilla: {acc_van:.3f}")
# XL climbs high (it attends back into the cached segment 1 to fetch the answer);
# vanilla plateaus near chance (~1/10) because segment 2 cannot see segment 1 at all.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_On a cross-segment echo task, does segment-level memory let the model fetch an answer from the previous segment, and does removing it (the vanilla segmented baseline) make that dependency unreachable?_

### Step 1 — Rebuild the XL model compactly

This panel stands alone, so we re-declare a compact version of the same XL attention, block, and segment-processing model — identical logic to the reference implementation, just tersely named. Re-seeding keeps the run reproducible.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class XLAttn(nn.Module):
    def __init__(self, d, max_rel=64):
        super().__init__()
        self.q, self.k, self.v, self.o = (nn.Linear(d, d, bias=False) for _ in range(4))
        self.d, self.max_rel = d, max_rel
        self.rel = nn.Parameter(torch.zeros(2 * max_rel + 1))
    def forward(self, x, mem=None):
        ctx = torch.cat([mem.detach(), x], 0) if mem is not None else x
        S, M = x.shape[0], ctx.shape[0] - x.shape[0]
        sc = self.q(x) @ self.k(ctx).transpose(0, 1) / math.sqrt(self.d)
        qi = torch.arange(S).unsqueeze(1) + M
        kj = torch.arange(M + S).unsqueeze(0)
        d = (qi - kj).clamp(-self.max_rel, self.max_rel) + self.max_rel
        a = F.softmax(sc + self.rel[d], -1)
        return self.o(a @ self.v(ctx))

class Block(nn.Module):
    def __init__(self, d, ff):
        super().__init__()
        self.a = XLAttn(d)
        self.f = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x, mem=None):
        x = self.n1(x + self.a(x, mem))
        return self.n2(x + self.f(x))

class XL(nn.Module):
    def __init__(self, V, d=32, ff=64):
        super().__init__()
        self.e = nn.Embedding(V, d)
        self.b = Block(d, ff)
        self.o = nn.Linear(d, V)
    def forward(self, segs, use_mem=True):
        mem, out = None, []
        for s in segs:
            h = self.b(self.e(s), mem if use_mem else None)
            out.append(self.o(h))
            mem = h
        return out

### Step 2 — Train both variants and record the curve

We rebuild the same echo task and train memory-on and memory-off, recording cue accuracy at every step. Memory on climbs toward ~1.0 (it fetches the answer from cached segment 1); memory off stays flat near chance ~0.1, the cross-segment dependency unreachable.

In [ ]:
V, L, B = 12, 4, 256
CUE = V - 1

def batch():
    s1 = torch.randint(1, CUE, (B, L))
    s1[:, 0] = CUE
    ans = s1[:, 1]
    s2 = torch.randint(1, CUE, (B, L))
    t2 = s2.clone()
    t2[:, -1] = ans
    return s1, s2, t2

def run(use_mem, steps=600):
    torch.manual_seed(0)
    net = XL(V)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    lf = nn.CrossEntropyLoss()
    accs = []
    for st in range(steps):
        s1, s2, t2 = batch()
        opt.zero_grad()
        loss = 0.0
        acc = 0.0
        for b in range(B):
            lg = net([s1[b], s2[b]], use_mem=use_mem)[1]
            loss = loss + lf(lg, t2[b])
            acc += (lg[-1].argmax() == t2[b, -1]).float().item()
        (loss / B).backward()
        opt.step()
        accs.append(acc / B)
    return accs

on = run(True)
off = run(False)
idx = list(range(0, 600, 75)) + [599]
print("memory on :", [[i, round(on[i], 3)] for i in idx])
print("memory off:", [[i, round(off[i], 3)] for i in idx])
# memory on -> climbs to ~1.0 (fetches the answer from cached segment 1).
# memory off -> flat near chance ~0.1 (segment 2 cannot see segment 1: context fragmentation).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your model solves the cross-segment echo task (it reaches high accuracy) when
            segment 2 can attend into the cached segment 1. Turn the memory off &mdash; pass an empty
            cache to segment 2, which is exactly the vanilla segmented baseline of &sect;3.1 &mdash; and
            retrain. What happens to accuracy on the cross-segment echo, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: feed segment 2 an empty memory instead of segment 1's cached states; keep depth, width, heads, optimizer, data, and seed identical. — _An honest ablation isolates the segment memory &mdash; any difference is attributable to it, not to capacity or tuning._
- Retrain and watch accuracy on the cued token: with memory it climbs high; without it, accuracy on the cross-segment cue collapses toward chance while in-segment predictions are unaffected. — _The answer to the echo lives in the previous segment; with no cache, segment 2 has no path to it &mdash; this is the &sect;3.1 context-fragmentation problem made concrete._
- Conclude that the segment-level recurrence, not extra parameters, is what carries the long-range dependency across the boundary. — _Both runs share architecture and parameter count; only the cache differs, isolating it as the cause._

**Answer:** With the memory removed, the model can no longer solve the cross-segment echo: accuracy on the
                 cued token drops toward chance, because the information it needs sits in the previous
                 segment and the vanilla baseline throws that segment away (the context-fragmentation problem,
                 &sect;3.1). In-segment predictions are unchanged. Since the two runs are identical except for the
                 cache, this isolates segment-level recurrence as the source of the long-range capability. The
                 CODEVIZ panel shows exactly this contrast.

</details>

**Problem 2.** In the worked example the memory model output $\approx 3.964$ for the current token, while the
            no-memory version output exactly $2.0$. Trace where the extra pull toward $4$ came from, and what it
            tells you about how far the model is "seeing."

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the current token's hidden state is $2$, and the cached segment held values $[4,-1]$. — _Only the memory model can attend to those cached values; the vanilla model sees only the current token._
- Compute the attention weights of the query ($q=2$) over the extended keys $[4,-1,2]$: scores $[8,-2,4]$, softmax $\approx[0.982,0.000,0.018]$. — _The large score $8$ on the cached value $4$ dominates the softmax, so almost all the weight lands on a token from the previous segment._
- Read the output $0.982{\cdot}4 + 0.018{\cdot}2 \approx 3.964$ and compare to the vanilla $2.0$. — _The shift from $2$ toward $4$ is information fetched across the segment boundary &mdash; impossible without the cache._

**Answer:** The query (hidden state $2$) scores highest against the cached value $4$ (score $8$), so the
                 softmax puts $\approx 98\%$ of its weight there and the output is pulled to $\approx 3.964$.
                 The vanilla model, with no cache, can only attend to the single current token and returns $2.0$.
                 The gap is exactly the information the memory model pulled from the previous segment &mdash;
                 a dependency longer than one window, which is the paper's whole point.

</details>

**Problem 3.** Transformer-XL replaces the original absolute positional encoding with a relative one
            (&sect;3.3). Explain concretely what goes wrong if you keep absolute positions once you start
            caching the previous segment, and how the relative score fixes it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that with absolute encoding, token $i$ gets a position vector $U_i$ added by its index within its window. — _Both the cached segment and the current segment are indexed from $0$, so position $0$ of each gets the same $U_0$._
- Concatenate the two segments and observe that the model can no longer tell a cached token apart from a current token at the same within-window index. — _Identical position vectors make the splice ambiguous &mdash; the model loses track of which segment a token belongs to._
- Switch to the relative score $A_{i,j}^{\mathrm{rel}}$ where position enters only through $R_{i-j}$, the distance between query and key. — _Distance is well-defined across the boundary: a cached key is simply 'further to the left,' so the score stays coherent (\S 3.3)._

**Answer:** Absolute encoding indexes each segment from $0$, so after caching, position $0$ of the previous
                 segment and position $0$ of the current segment carry the identical vector $U_0$ &mdash; the model
                 cannot tell spliced tokens apart, and the recurrence becomes incoherent. The relative score makes
                 position enter only as the distance $R_{i-j}$ between query and key (plus the learned biases
                 $u,v$), which is unambiguous across the boundary: a cached token is just "$k$ positions to the
                 left." That is why segment recurrence and relative positions are introduced together.

</details>